# 🔘🖼️ Gradio Advanced

⚠️ **Remember to copy this notebook in your Drive and rename.**

*Workflows for IAAC MaCDA GenAI  (Apr - Jun 2026) taught by [James McBennett](https://www.linkedin.com/in/mcbennett/) and [Aymeric Brouez](https://www.linkedin.com/in/aymeric-brouez/)*

*With special thanks to past faculty [Nono Martínez Alonso](https://youtube.com/NonoMartinezAlonso).*

**🗒️ Gradio's [Documentation](https://www.gradio.app/docs)**

Gradio's [Documentation](https://www.gradio.app/docs)

In [ ]:
import torch
print(f"VRAM Total: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")

## Install Gradio (and Restart)

In [ ]:
!pip install gradio --quiet

### Mount Drive

In [ ]:
# Mount Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

## Setup

In [ ]:
%cd /content
!rm -rf iaac_genai
!git clone https://github.com/jamesmcbennett/iaac_genai
%cd /content/iaac_genai/

In [ ]:
import sys
sys.path.append('/content/iaac_genai')

In [ ]:
!pip install -q -r requirements.txt --quiet > /dev/null 2>&1

In [ ]:
from config import Config
from utils import set_image_path, save_image, save_yml, save_svg
import torch

from diffusers.utils import load_image
from PIL import Image, ImageDraw, ImageFont
import gradio as gr
import numpy as np
import cv2
import math

## Prompt Construction

In [ ]:
# users can edit prompt_edit but start and end are automatically added on either end

prompt_start = "jmsmcbnntt trigger word, an architectural model of a "
prompt_edit = "Irish pub"
prompt_end = ", centered pure white background, no shadows, no texture, rendered in thin green linework, no surrounding context"

PROMPT = f"{prompt_start}{prompt_edit}{prompt_end}"
print(PROMPT)

## Sketch

In [ ]:
import gradio as gr
from PIL import Image
import numpy as np

# Save Path and Save Variable
SKETCH_PATH = "/content/sketch_output.png"
gradio_image = None

# Function
def save_sketch_and_store(editor_value):
    global gradio_image
    try:
        composite = editor_value["composite"]
        if isinstance(composite, np.ndarray):
            img = Image.fromarray(composite.astype(np.uint8), mode="RGBA")
        else:
            img = composite
        img.save(SKETCH_PATH)
        gradio_image = img
        return f"✅ Saved as 'gradio_image'"
    except Exception as e:
        return f"❌ Error: {e}"

# Gradio UI
with gr.Blocks() as demo:
    gr.Markdown("🎨 Draw with color, then save your sketch")
    editor = gr.ImageEditor(label="Sketch Editor", type="numpy", image_mode="RGBA")
    save_btn = gr.Button("💾 Save as 'gradio_image'")
    status = gr.Textbox(label="Status", lines=2)
    save_btn.click(save_sketch_and_store, inputs=editor, outputs=status)

demo.launch(share=True, debug=True)

Check image saved saved correctly

In [ ]:
gradio_image

## Vectorize

In [ ]:
!apt-get install potrace
!pip install svgwrite

In [ ]:
from PIL import Image
from IPython.display import SVG, display
import os

# --- SETUP PATHS ---
# INPUT_PATH = "/content/drive/MyDrive/iaac_genai/workflows/03_FINETUNING/02_FLUX.1/dataset_FLUX.1/bof_aar_lacha_02.png"
INPUT_PATH = "/content/drive/MyDrive/iaac_genai/inputs/dom_wilcox/1.jpg"


OUTPUT_DIR = "/content/drive/MyDrive/output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# --- LOAD AND DISPLAY IMAGE ---
def load_image(path):
    return Image.open(path)

input_image = load_image(INPUT_PATH)
input_image

In [ ]:
import re, os
# Convert to black and white, then save as PBM
# Examples Colours: "red", "green", "blue", "black", "white", "gray", "orange", "purple", "pink", "cyan", "teal", "brown"
# Examples Hex:     "#FF0000", "#00FF00", "#0000FF", "#000000", "#FFFFFF", "#808080", "#FFA500", "#800080", "#FFC0CB", "#00FFFF", "#008080", "#A52A2A"
# Example RGB:      "(255,0,0)", "(0,255,0)", "(0,0,255)", "(0,0,0)", "(255,255,255)", "(128,128,128)", "(255,165,0)", "(128,0,128)", "(255,192,203)", "(0,255,255)", "(0,128,128)", "(165,42,42)"
# Scale threshold: try between 180–230 depending on line weight

scale = 130
fill_color = "#6ecbfa"
stroke_color = "#2d9dd6"
stroke_width = 15

bw = input_image.convert("L").point(lambda x: 0 if x < scale else 255, '1')
pbm_path = os.path.join(OUTPUT_DIR, "vector_input.pbm")
svg_path = os.path.join(OUTPUT_DIR, "vector_output.svg")

bw.save(pbm_path)

# Run potrace to vectorize
!potrace "{pbm_path}" -s -o "{svg_path}"

# Run
svg_path = os.path.join(OUTPUT_DIR, "vector_output.svg")
style_str = f'style="fill:{fill_color};stroke:{stroke_color};stroke-width:{stroke_width}"'
with open(svg_path, "r") as f:
    svg_data = f.read()
svg_data = re.sub(r'<path[^>]*style="[^"]*"[^>]*>',
                  lambda m: re.sub(r'style="[^"]*"', style_str, m.group(0)),
                  svg_data)
svg_data = re.sub(r'<path(?![^>]*style=)', f'<path {style_str}', svg_data)
with open(svg_path, "w") as f:
    f.write(svg_data)

svg_path = os.path.join(OUTPUT_DIR, "vector_output.svg")
display(SVG(filename=svg_path))

# from google.colab import files
# files.download(svg_path)